# Context Coherence Score — interactive tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saket-Maganti/rag-hallucination-detection/blob/main/notebooks/colab_tutorial.ipynb)

This 6-cell notebook walks you through the **Context Coherence Score (CCS)** and the **refinement paradox** from Maganti (2026) — *"When Better Retrieval Hurts: Context Coherence Drives Faithfulness in RAG"*.

Total runtime: **< 5 minutes** on free Colab CPU. No GPU, no API keys.

**What you'll do:**
1. Install the standalone `context-coherence` package.
2. Compute CCS on a few example retrieval sets.
3. See the paradox visually using pre-computed paper data.
4. Apply the `CCSGate` decision policy in 3 lines.
5. Load the full ContextCoherenceBench from HF Datasets.
6. Reproduce one of the paper's headline plots.


In [ ]:
# Cell 1 — install (~30 s on Colab)
%pip install -q context-coherence sentence-transformers datasets pandas matplotlib


In [ ]:
# Cell 2 — compute CCS on three toy retrieval sets
from context_coherence import ccs
from sentence_transformers import SentenceTransformer

st = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

coherent  = ['The mitochondrion produces ATP.',
             'ATP is the cell\'s primary energy currency.',
             'Cellular respiration generates ATP in the mitochondria.']

fragmented = ['The mitochondrion produces ATP.',
              'Quantum entanglement is a fundamental physics phenomenon.',
              'The Eiffel Tower is in Paris.']

print(f'CCS coherent set:    {ccs(coherent,  embedder=st):+.3f}')
print(f'CCS fragmented set:  {ccs(fragmented, embedder=st):+.3f}')
print()
print('Higher CCS = more internally consistent retrieval set.')


In [ ]:
# Cell 3 — the refinement paradox in one image
# Pull the pre-computed paper data
import urllib.request, pandas as pd, matplotlib.pyplot as plt

url = 'https://raw.githubusercontent.com/Saket-Maganti/rag-hallucination-detection/main/results/multidataset/coherence_paradox.csv'
df = pd.read_csv(urllib.request.urlretrieve(url)[0])
df.head()


In [ ]:
# Cell 4 — the CCS gate as a deployment policy
from context_coherence import CCSGate

gate = CCSGate(threshold=0.5)

for name, passages in [('coherent', coherent), ('fragmented', fragmented)]:
    decision = gate.decision(passages, embedder=st)
    action = 're-retrieve / fall back' if decision['fires'] else 'pass to generator'
    print(f'{name:12s}  CCS={decision["ccs"]:+.3f}  -> {action}')


In [ ]:
# Cell 5 — load the full benchmark from HuggingFace Datasets
from datasets import load_dataset

# Adversarial coherence cases (4 splits: drift / disjoint / contradict / control)
drift = load_dataset('saketmgnt/context-coherence-bench',
                     'adversarial_drift', split='train')
print(f'Loaded {len(drift)} drift cases.')
print('First case:')
print(drift[0])


In [ ]:
# Cell 6 — reproduce the headline paradox bar chart
import matplotlib.pyplot as plt

datasets = ['squad', 'pubmedqa']
models = df['model'].unique().tolist()[:3]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, ds in zip(axes, datasets):
    sub = df[df['dataset'] == ds]
    if sub.empty:
        ax.set_title(f'{ds} (no rows)'); continue
    sub.plot.bar(x='model', y=['baseline_faith', 'hcpc_v1_faith', 'hcpc_v2_faith'],
                  ax=ax, rot=20)
    ax.set_title(ds.upper()); ax.set_ylabel('faithfulness'); ax.legend(fontsize=7)
fig.tight_layout()
plt.show()

print('Notice: HCPC-v1 (orange) DROPS faithfulness vs baseline (blue) on most rows.')
print('That is the refinement paradox. HCPC-v2 (green) recovers it.')


## Where to go next

- **Standalone metric**: `pip install context-coherence` for the CCS metric in your own pipeline.
- **Full repo + paper**: <https://github.com/Saket-Maganti/rag-hallucination-detection>
- **Permanent benchmark DOI**: <https://doi.org/10.5281/zenodo.19757291>
- **Interactive demo**: <https://huggingface.co/spaces/saketmgnt/sakkk>
- **Cite**: see `CITATION.bib` in the repo.
